# 02 - Character Segmentation
En aquest notebook ens centrem a extreure els caràcters individuals d'una matrícula que ja ha estat prèviament localitzada, retallada i redreçada.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def mostrar_imatge(titol, imatge, cmap=None):
    plt.figure(figsize=(10, 4))
    plt.title(titol)
    if cmap:
        plt.imshow(imatge, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(imatge, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Carregar la imatge original
plate_image = cv2.imread('data/plate_output.jpg')

mostrar_imatge("Input: Imatge amb il·luminació irregular", plate_image)

## Pas 1: Adaptive Threshold
Com que la il·luminació dins del retall de la matrícula podria no ser perfecta (per exemple, l'ombra del para-xocs), l'ús de l'Adaptive Threshold ens garanteix que totes les lletres es binaritzin correctament independentment dels gradients de llum.

In [ ]:
# Convertim a grisos
gray = cv2.cvtColor(plate_image, cv2.COLOR_BGR2GRAY)

# Adaptive Threshold (Invertit, perquè busquem objectes blancs sobre fons negre)
thresh = cv2.adaptiveThreshold(
    gray, 
    255, 
    cv2.ADAPTIVE_THRESH_MEAN_C, 
    cv2.THRESH_BINARY_INV, 
    31,  # Mida de la finestra (ha de ser senar)
    15   # Offset
)

mostrar_imatge("1. Adaptive Threshold (Binarització)", thresh, cmap='gray')

## Pas 2: CCA (Contour)
En comptes de fer servir connectedComponents, OpenCV ens ofereix findContours, que és molt eficient per trobar els blocs de text sobre la màscara binaritzada. Utilitzarem RETR_EXTERNAL per ignorar els forats de lletres com la 'O' o la 'D' (només volem la capsa exterior).

In [ ]:
# Trobem contorns externs
cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

img_copy = plate_image.copy()
for c in cnts:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(plate_image, (x, y), (x+w, y+h), (0, 0, 255), 2)

mostrar_imatge("2. Contorns detectats (incloent soroll)", plate_image)

# Extraiem els Bounding Boxes de tots els contorns trobats
tots_els_bboxes = [cv2.boundingRect(c) for c in cnts]

print(f"S'han detectat {len(tots_els_bboxes)} contorns en total (incloent soroll).")

## Pas 3: Filter (Stricter on height variance)

Sabem que en una matrícula autèntica, tots els caràcters tenen exactament la mateixa alçada (pot haver-hi una variació mínima de 1 o 2 píxels).
El que farem serà:

Descartar la brutícia minúscula per no esbiaixar les matemàtiques.

Calcular la Mediana d'Alçada (Median Height) de la resta d'elements.

Acceptar només els blobs que la seva alçada es desviï com a màxim un ±15% d'aquesta mediana.

In [ ]:
img_filtrada = plate_image.copy()
chars_filtrats = []

# 1. Calculem la mediana d'alçada d'aquells blocs que tinguin una alçada mínima 
#    (Això ignora pols i petits detalls que desquadrarien la mediana)
altures_valides = [h for (x, y, w, h) in tots_els_bboxes if h > 10]
print(f"Alçades vàlides per a càlcul de mediana: {len(altures_valides)}")

if altures_valides:
    h_mediana = np.median(altures_valides)
    print(f"Alçada Mediana de referència: {h_mediana} píxels.")
    
    # 2. Apliquem el filtre estricte (Stricter on height variance)
    for (x, y, w, h) in tots_els_bboxes:
        aspect_ratio = w / float(h)
        
        # Filtre d'Alçada: L'alçada ha de ser entre el 85% i el 115% de la mediana
        # Filtre d'Amplada: Evitem lletres fusionades (massa amples) o soroll vertical
        if (h_mediana * 0.85 < h < h_mediana * 1.15) and (0.15 < aspect_ratio < 0.95):
            chars_filtrats.append((x, y, w, h))
            cv2.rectangle(img_filtrada, (x, y), (x+w, y+h), (0, 255, 0), 2)

# Ordenem d'esquerra a dreta
chars_filtrats.sort(key=lambda b: b[0])

mostrar_imatge("3. Filtre Estricte (Cargols i marges ignorats)", img_filtrada)
print(f"Caràcters finals retinguts: {len(chars_filtrats)}")

## Pas 4: Copy and Resize Characters
L'últim pas del processament d'imatge abans del Machine Learning. Recorrem els caràcters filtrats, en "copiem" el retall de la imatge binaritzada, i els forcem a tenir una resolució constant perquè la CNN els pugui processar en format batch.

In [ ]:
# Dimensions típiques d'entrada per a CNNs OCR (per ex. EMNIST)
# A vegades es fa servir 28x28 (quadrat) o 32x64 (rectangular)
CNN_INPUT_W = 28
CNN_INPUT_H = 28

caracters_per_cnn = []

fig, axes = plt.subplots(1, len(chars_filtrats), figsize=(12, 3))
if len(chars_filtrats) == 1:
    axes = [axes]

for idx, (x, y, w, h) in enumerate(chars_filtrats):
    # COPY: Retallem des de la imatge threshold (binaritzada)
    # Les Xarxes Neuronals de text solen preferir el text blanc sobre fons negre
    char_crop = thresh[y:y+h, x:x+w]
    
    # RESIZE: El redimensionem
    char_resized = cv2.resize(char_crop, (CNN_INPUT_W, CNN_INPUT_H), interpolation=cv2.INTER_AREA)
    
    caracters_per_cnn.append(char_resized)
    
    axes[idx].imshow(char_resized, cmap='gray')
    axes[idx].axis('off')
    axes[idx].set_title(f"Char {idx+1}")

plt.suptitle("4. Copy and Resize (Ready for CNN)")
plt.show()

# Ara `caracters_per_cnn` és una llista de matrius NumPy llistes per inferència:
# model.predict(np.array(caracters_per_cnn))

## Pas 5: Guardar els caràcters al disc
Finalment, crearem una carpeta a la ruta data/chars/ per emmagatzemar els caràcters segmentats. Aquest pas és essencial per poder carregar les imatges més tard durant l'entrenament o la inferència de la nostra xarxa neuronal. Cada imatge es guardarà amb un índex seqüencial per mantenir l'ordre de la matrícula.

In [ ]:
import os

# 1. Definir la ruta de sortida
output_path = 'data/chars/'

# 2. Crear la carpeta si no existeix (makedirs crea carpetes pare si cal)
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"S'ha creat el directori: {output_path}")
else:
    # Opcional: Netejar la carpeta si ja existia per evitar barrejar matrícules
    for file in os.listdir(output_path):
        os.remove(os.path.join(output_path, file))
    print(f"El directori ja existeix. S'han netejat els fitxers anteriors.")

# 3. Guardar cada caràcter individualment
for idx, char_img in enumerate(caracters_per_cnn):
    # Generem el nom del fitxer (p.ex. char_0.png, char_1.png...)
    file_name = f"char_{idx}.png"
    full_file_path = os.path.join(output_path, file_name)
    
    # Guardem la imatge (OpenCV usa format uint8)
    success = cv2.imwrite(full_file_path, char_img)
    
    if success:
        print(f"S'ha desat: {full_file_path}")
    else:
        print(f"Error en desar: {full_file_path}")

print("\nProcés finalitzat. Tots els caràcters estan llestos a 'data/chars/'.")